In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
#custom class
import os
import torch
from torch.utils.data import Dataset
import glob
from PIL import Image



class CustomClass(Dataset):

  def __init__(self , root_dir , split = 'train', transform = None):
    self.root_dir = root_dir
    self.split = split
    self.transform = transform
    self.class_labels = {
            "Early": 0, "Late": 1, "healthy": 2,
    }

     #Extract paths:

    images_paths = []
    lable_paths = []

    joined_path =  os.join.path(self.root_dir , self.split)

    # Get all image paths
    self.image_paths = []
    self.labels = []
    for class_name, label in self.class_labels.items():
      class_images = glob.glob(f"{root_dir}/{class_name}/*.JPG")  # Find all images
      self.image_paths.extend(class_images)
      self.labels.extend([label] * len(class_images))  # Assign labels



  def __len__(self):
    return len(self.images_paths)

  def __getitem__(self, idx):
     image_path = self.image_paths[idx]  # Get image path
     label = self.labels[idx]  # Get label

     # Load image using PIL
     image = Image.open(image_path)

        # Apply transformations (if any)
     if self.transform:
          image = self.transform(image)

     return image, label  # Return processed image & label


In [ ]:
#create dataset:

from torchvision import transforms
from torch.utils.data import DataLoader

# Define transformations
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images
    transforms.RandomRotation(15),  # Rotate images randomly within ±15 degrees
    transforms.ToTensor(),  # Convert to tensor
])

# Validation and testing data typically don’t require augmentations, as we only evaluate the model performance on these sets.
# Instead, we apply basic transformations to prepare the images.
transform_valid_test = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images to 32*32
    transforms.ToTensor(),  # Convert to tensor
])


# Initialize dataset for Train
train_path = os.path.join(path, split="train")
test_path = os.path.join(path, "test")

train_dataset = CustomClass(train_path, transform=transform)
test_dataset = CustomClass(test_path, transform=transform_valid_test)

# Create DataLoader
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

# Get a batch of training images
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels}")

In [ ]:
# Write your code here


import torch
import torch.nn as nn

class CustomModel(nn.Module):
    def __init__(self):
        """
        1️⃣ Define all layers in the model.
        """
        super(CustomModel, self).__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channel , 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )


        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        self.layer3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )


        self.layer4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )

        self.enc3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        self.layer5 = nn.Sequential(
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True),
            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True)
        )


        # Fully Connected Layer
        self.fc = nn.Linear(16 * 16 * 16, 10)  # Output 10 classes

        # Softmax Layer
        self.softmax = nn.Softmax(dim=1)  # Apply along the class dimension

    def forward(self, x):
        """
        2️⃣ Define the forward pass (how data flows through the model).
        """
        x = self.conv1(x)  # Convolution
        x = self.relu(x)  # Activation
        x = self.pool(x)  # Pooling
        x = torch.flatten(x, start_dim=1)  # Flatten for FC layer
        x = self.fc(x)  # Fully connected layer
        x = self.softmax(x)  # Convert logits to probabilities
        return x  # Returns probability distribution


In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode, you will understand why later
    total_loss = 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)  # Move data to GPU if available


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss


        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation (compute gradients)
        optimizer.step()  # Update model parameters

        # Collect the loss
        total_loss += loss.item()

    return total_loss / len(dataloader)  # Return average loss


In [ ]:
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode, you will understand why later
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient calculation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # (Optional) Compute accuracy
            predictions = outputs.argmax(dim=1)  # Get class with highest probability
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


In [ ]:
# Write your code here

import torch.optim as optim


# Run Training
model = CustomModel(nn.Module)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):  # Train for 5 epochs
    train_one_epoch(model, train_loader, criterion, optimizer, device)
    accuracy = validate(model, test_loader, criterion, device)
    print(f"Epoch {epoch+1}: Validation Accuracy = {accuracy:.2f}%")

In [ ]:
# Write your code here
